### tests/test_fk.ipynb
- tests/test_fk.py

In [ ]:
%run -i ../src/transforms.ipynb
%run -i ../src/fk_chain.ipynb

In [ ]:
import numpy as np
import pytest

arm = PlanarArm([0.3, 0.2])
CASES = [((0, 0), (0.5, 0.0)),
         ((np.pi/2, 0), (0.0, 0.5)),
         ((0, np.pi/2), (0.3, 0.2)),
         ((np.pi/2, -np.pi/2), (0.2, 0.3)),
         ((np.pi/4, np.pi/4), (0.3*np.cos(np.pi/4), 0.3*np.sin(np.pi/4)+0.2))]

In [ ]:
@pytest.mark.parametrize("thetas, expected", CASES)
def test_fk_known_poses(thetas, expected):
    np.testing.assert_allclose(arm.fk(thetas), expected, atol=1e-12)

In [ ]:
def test_fk_3dof_matches_formula():
    arm3 = PlanarArm([0.3, 0.2, 0.1])
    th = np.array([0.3, -0.4, 0.9])
    cum = np.cumsum(th)
    expected = (np.sum(arm3.L * np.cos(cum)), np.sum(arm3.L * np.sin(cum)))
    np.testing.assert_allclose(arm3.fk(th), expected, atol=1e-12)

In [ ]:
# 노트북 안에서 테스트 실행 (pytest.mark.parametrize 케이스까지 전개)
_passed = 0
for _name, _fn in list(globals().items()):
    if not (_name.startswith("test_") and callable(_fn)):
        continue
    _ps = [m for m in getattr(_fn, "pytestmark", []) if m.name == "parametrize"]
    if _ps:
        _argnames = [a.strip() for a in _ps[0].args[0].split(",")]
        for _vals in _ps[0].args[1]:
            _fn(*(_vals if len(_argnames) > 1 else (_vals,)))
            _passed += 1
    else:
        _fn(); _passed += 1
    print("PASS", _name)
print(f"{_passed} passed")